# Direct-CP toy fit for $B^\pm\to K^\pm\pi^+\pi^-$

`generate_cp_toy` performs the charge split from the accepted amplitude integrals and `CPFitSession` builds the joint charge-Dalitz likelihood automatically.


In [ ]:
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    CPFitSession, CPRealImag, DecayChannel, DecayModel, NonResonant,
    Parameter, Resonance, enable_x64, generate_cp_toy, plot_dalitz,
)
enable_x64()


In [ ]:
shared = {
    "Kstar": CPRealImag(
        Parameter.coefficient("Kstar.x",1.0,fixed=True,owner="Kstar"),
        Parameter.coefficient("Kstar.y",0.0,fixed=True,owner="Kstar"),
        Parameter.coefficient("Kstar.dx",0.04,bounds=(-0.3,0.3),owner="Kstar",step=0.01),
        Parameter.coefficient("Kstar.dy",-0.03,bounds=(-0.3,0.3),owner="Kstar",step=0.01),
    ),
    "rho": CPRealImag(
        Parameter.coefficient("rho.x",0.60,bounds=(-2,2),owner="rho",step=0.02),
        Parameter.coefficient("rho.y",0.15,bounds=(-2,2),owner="rho",step=0.02),
        Parameter.coefficient("rho.dx",0.05,bounds=(-0.5,0.5),owner="rho",step=0.01),
        Parameter.coefficient("rho.dy",0.02,bounds=(-0.5,0.5),owner="rho",step=0.01),
    ),
}

def components(q):
    return [
        Resonance("Kstar",(0,2),shared["Kstar"].for_charge(q),
                  mass=0.8958,width=0.0474,spin=1),
        Resonance("rho",(1,2),shared["rho"].for_charge(q),
                  mass=0.7753,width=0.1491,spin=1),
        NonResonant(CPRealImag(-0.30,0.10,0.0,0.0).for_charge(q)),
    ]

plus_model = DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),components(+1),
    normalization_method="square-dalitz",normalization_resolution=220,
    normalization_pair=(0,2),
)
minus_model = DecayModel(
    DecayChannel("B-",("K-","pi-","pi+")),components(-1),
    normalization_method="square-dalitz",normalization_resolution=220,
    normalization_pair=(0,2),
)
truth={p.name:p.value for p in plus_model.parameters}


In [ ]:
plus_data, minus_data = generate_cp_toy(
    plus_model, minus_model, 40_000, parameters=truth,
    seed=505, pool_size=250_000,
)
print("B+ / B-:", plus_data.size, minus_data.size)

fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
plot_dalitz(plus_data,x="s13",y="s23",ax=axes[0],title="B+")
plot_dalitz(minus_data,x="s13",y="s23",ax=axes[1],title="B-")
plt.show()


In [ ]:
session = CPFitSession(plus_model,minus_model,plus_data,minus_data)
start={p.name:p.value+0.06 for p in session.parameters if not p.fixed}
result=session.fit(start,simplex=True,ncall=50_000)
session.report(result)
session.plot_projection(result,"s13")
plt.show()
